# AR, MA, ARIMA & SARIMA — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## Forecasting by regressing on past values, past shocks, differences, and seasonal differences
ARIMA models organize dependence into four knobs: autoregression, moving-average shocks, differencing, and seasonality. These examples isolate each knob on tiny synthetic series, then combine them in a seasonal forecast.

In [ ]:
# 1) AR(1): each forecast carries 60% of the previous value
phi=0.6; y=[2.0]
for _ in range(4): y.append(phi*y[-1])
plt.figure(figsize=(6,3)); plt.plot(range(5),y,'o-'); plt.title('AR(1) impulse decays geometrically')
assert np.allclose(y,[2.0,1.2,0.72,0.432,0.2592])

In [ ]:
# 2) MA(1): today's value mixes today's shock and yesterday's shock
eps=np.array([1.0,-0.5,0.2]); theta=0.4; y=np.array([eps[0],eps[1]+theta*eps[0],eps[2]+theta*eps[1]])
plt.figure(figsize=(6,3)); plt.bar(range(3),y); plt.title('MA(1) shock memory lasts one step')
assert np.allclose(y,[1.0,-0.1,0.0])

In [ ]:
# 3) differencing: ARIMA removes a changing level before ARMA modeling
y=np.array([3.,4.,6.,9.]); dy=np.diff(y)
plt.figure(figsize=(6,3)); plt.plot(y,'o-',label='level'); plt.plot(np.arange(1,4),dy,'s-',label='difference'); plt.legend(); plt.title('differences are increments: 1,2,3')
assert np.allclose(dy,[1.,2.,3.])

In [ ]:
# 4) seasonal differencing removes a repeated yearly/monthly pattern
base=np.arange(16)*0.1; season=np.tile([2,0,-2,0],4); y=10+base+season; D=y[4:]-y[:-4]
plt.figure(figsize=(6,3)); plt.plot(y,label='seasonal series'); plt.plot(np.arange(4,16),D,label='lag-4 difference'); plt.legend(); plt.title('SARIMA uses seasonal differences')
assert np.allclose(D,0.4*np.ones(12))

In [ ]:
# 5) tiny SARIMA-style forecast: seasonal naive plus AR correction
hist=np.array([10,12,9,11,10.5,12.5,9.4,11.2]); seasonal_base=hist[-4]; residuals=hist[4:]-hist[:4]; correction=0.5*residuals[-1]; forecast=seasonal_base+correction
plt.figure(figsize=(6,3)); plt.plot(hist,'o-',label='history'); plt.scatter([8],[forecast],c='r',label='forecast'); plt.legend(); plt.title(f'forecast={forecast:.2f}')
assert abs(forecast-10.6)<1e-9